# ShallowSeek - KD Draft Evaluation for Speculative Decoding

**Zhiyan Ke - 407061**

This notebook is my individual Part II milestone notebook. It is intended to run inside the RunAI/RCP Jupyter environment launched by `notebooks/submit.sh`.

It demonstrates my current contribution to the project:

- loading the Hugging Face UltraChat dataset and Qwen draft model;
- loading the KD-trained draft checkpoint from my RunAI scratch directory when available;
- collecting preliminary speculative-decoding results for pretrained, FKL, RKL, and JSD drafts;
- comparing acceptance rate, average accepted tokens, and measured wall-clock speedup;
- recording the exact evaluation commands needed to reproduce the results.

The expensive evaluation commands are printed by default rather than executed. Set `RUN_EVAL=True` below to run them in the interactive container.

## 1. Environment and Paths

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "evaluate_sd.py").exists() and (p / "src" / "kdsd").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
SCRATCH_ROOT = Path("/scratch/cs552-mnlp-kzy")
CHECKPOINTS_DIR = SCRATCH_ROOT / "checkpoints"
RESULTS_DIR = SCRATCH_ROOT / "results"
DATA_DIR = SCRATCH_ROOT / "data"
HF_CACHE = SCRATCH_ROOT / "hf_cache"

os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_HUB_CACHE", str(HF_CACHE / "hub"))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE / "datasets"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("repo root:", REPO_ROOT)
print("scratch root:", SCRATCH_ROOT)
print("checkpoints exist:", CHECKPOINTS_DIR.exists())
print("results exist:", RESULTS_DIR.exists())
print("HF_HOME:", os.environ.get("HF_HOME"))

In [ ]:
# Execution switches.
# Keep RUN_EVAL=False for a fast notebook demo. Full eval can take a long time.
RUN_EVAL = False
LOAD_PRETRAINED_DRAFT = True
LOAD_TRAINED_FKL_CHECKPOINT = True

# Shared evaluation protocol used for the 50k/8000-step runs.
DATASET = "ultrachat_50k"
EVAL_JSONL = DATA_DIR / "processed" / DATASET / "eval.jsonl"
EVAL_LIMIT = 50
GAMMA = 4
MAX_NEW_TOKENS = 256
N_WARMUP = 1
N_REPEATS = 3
RUNTIME_MODE = "sampling"
RUNTIME_TEMPERATURE = 1.0
TOP_P = 0.9

TARGET_ID = "Qwen/Qwen2.5-3B-Instruct"
PRETRAINED_DRAFT_ID = "Qwen/Qwen2.5-0.5B-Instruct"

RUN_SUFFIX = "ultra50k_s8000_seq512_a1_temp2"

print("RUN_EVAL =", RUN_EVAL)
print("eval prompts:", EVAL_JSONL, "exists=", EVAL_JSONL.exists())

## 2. Load Hugging Face Dataset and Base Draft Model

The milestone asks each individual notebook to load a model and dataset from Hugging Face. This section loads a small UltraChat slice and the pretrained Qwen2.5-0.5B draft model used as the initialization for our KD drafts.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

hf_sample = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:3]")
print(hf_sample)
print("first row keys:", list(hf_sample[0].keys()))

tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_DRAFT_ID, trust_remote_code=False)
print("tokenizer vocab size:", len(tokenizer))

pretrained_draft = None
if LOAD_PRETRAINED_DRAFT:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    pretrained_draft = AutoModelForCausalLM.from_pretrained(
        PRETRAINED_DRAFT_ID,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        attn_implementation="sdpa",
        trust_remote_code=False,
    ).to(device)
    print("loaded pretrained draft on", device)
    print("draft params (M):", round(sum(p.numel() for p in pretrained_draft.parameters()) / 1e6, 1))

## 3. Optionally Load My KD-Trained FKL Checkpoint

This section loads the checkpoint from my RunAI scratch directory if it exists. The notebook remains runnable if the checkpoint is absent, but in my interactive container it should load `/scratch/cs552-mnlp-kzy/checkpoints/fkl_ultra50k_s8000_seq512_a1_temp2/model`.

In [ ]:
FKL_CKPT = CHECKPOINTS_DIR / "fkl_ultra50k_s8000_seq512_a1_temp2" / "model"
trained_fkl_draft = None

if LOAD_TRAINED_FKL_CHECKPOINT and FKL_CKPT.exists():
    print("Loading trained FKL checkpoint:", FKL_CKPT)
    trained_fkl_draft = AutoModelForCausalLM.from_pretrained(
        str(FKL_CKPT),
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        attn_implementation="sdpa",
        trust_remote_code=False,
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    print("loaded trained FKL draft")
    print("checkpoint params (M):", round(sum(p.numel() for p in trained_fkl_draft.parameters()) / 1e6, 1))
else:
    print("FKL checkpoint not found or loading disabled:", FKL_CKPT)

In [ ]:
# Quick sanity generation with the trained checkpoint if loaded; otherwise with the pretrained draft.
model_for_demo = trained_fkl_draft or pretrained_draft
if model_for_demo is not None:
    model_for_demo.eval()
    prompt = "Explain speculative decoding in one sentence."
    if getattr(tokenizer, "chat_template", None):
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = prompt
    inputs = tokenizer(text, return_tensors="pt").to(model_for_demo.device)
    with torch.no_grad():
        out = model_for_demo.generate(**inputs, max_new_tokens=32, do_sample=False)
    print(tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))
else:
    print("No model loaded for sanity generation.")

## 4. Load Existing Evaluation Results

The full SD evaluation is slow, so this notebook reads the `eval_summary.json` files already produced on RunAI. The same protocol was used for the 50k/8000-step objective comparison: UltraChat-50k eval prompts, 50 prompts, `gamma=4`, `max_new_tokens=256`, sampling with `temperature=1.0`, `top_p=0.9`, one warmup, and three repeats.

In [ ]:
import pandas as pd

SUMMARY_PATHS = {
    "pretrained": RESULTS_DIR / "eval_pretrained_ultra50k_s8000_seq512_a1_temp2" / "eval_summary.json",
    "fkl": RESULTS_DIR / "eval_fkl_ultra50k_s8000_seq512_a1_temp2" / "eval_summary.json",
    "rkl": RESULTS_DIR / "eval_rkl_ultra50k_s8000_seq512_a1_temp2" / "eval_summary.json",
    "jsd": RESULTS_DIR / "eval_jsd_ultra50k_s8000_seq512_a1_temp2" / "eval_summary.json",
    "ce_10k": RESULTS_DIR / "eval_kd_ce_ultrachat10k_processed_bf16_seed42_g4_max256_p20" / "eval_summary.json",
    "fkl_10k": RESULTS_DIR / "eval_kd_fkl_ultrachat10k_processed_bf16_seed42_g4_max256_p20" / "eval_summary.json",
    "rkl_10k": RESULTS_DIR / "eval_kd_rkl_ultrachat10k_processed_bf16_seed42_g4_max256_p20" / "eval_summary.json",
    "jsd_10k": RESULTS_DIR / "eval_kd_jsd_ultrachat10k_processed_bf16_seed42_g4_max256_p20" / "eval_summary.json",
    "pretrained_10k_protocol": RESULTS_DIR / "eval_pretrained_qwen25_0p5b_ultrachat10k_processed_g4_max256_p20" / "eval_summary.json",
}

def load_summary(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

rows = []
for name, path in SUMMARY_PATHS.items():
    summary = load_summary(path)
    if summary is None:
        rows.append({"run": name, "status": "missing", "summary_path": str(path)})
        continue
    decoding = summary.get("decoding", {})
    rows.append({
        "run": name,
        "status": "done",
        "draft": summary.get("draft"),
        "n_prompts": summary.get("n_prompts"),
        "n_repeats": summary.get("n_repeats"),
        "gamma": decoding.get("num_assistant_tokens"),
        "max_new_tokens": decoding.get("max_new_tokens"),
        "acceptance_rate": summary.get("acceptance_rate"),
        "avg_accepted_tokens": summary.get("avg_accepted_tokens"),
        "speedup": summary.get("speedup"),
        "tokens_per_second": summary.get("tokens_per_second"),
        "sd_time_s": summary.get("sd_time_s"),
        "vanilla_time_s": summary.get("vanilla_time_s"),
        "summary_path": str(path),
    })

df = pd.DataFrame(rows)
df

In [ ]:
# Compact table for completed 50k objective runs.
completed = df[df["status"] == "done"].copy()
objective_order = ["pretrained", "fkl", "rkl", "jsd"]
obj50 = completed[completed["run"].isin(objective_order)].copy()
obj50["run"] = pd.Categorical(obj50["run"], objective_order, ordered=True)
obj50 = obj50.sort_values("run")

display_cols = [
    "run", "n_prompts", "n_repeats", "gamma", "max_new_tokens",
    "acceptance_rate", "avg_accepted_tokens", "speedup", "tokens_per_second",
    "sd_time_s", "vanilla_time_s",
]
obj50[display_cols]

## 5. Visualize the Preliminary Objective Comparison

In [ ]:
import matplotlib.pyplot as plt

if not obj50.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = [
        ("acceptance_rate", "Acceptance Rate"),
        ("avg_accepted_tokens", "Avg Accepted Tokens"),
        ("speedup", "Wall-clock Speedup"),
    ]
    for ax, (metric, title) in zip(axes, metrics):
        ax.bar(obj50["run"].astype(str), obj50[metric].astype(float), color=["#777777", "#4C72B0", "#55A868", "#DD8452"])
        if "pretrained" in obj50["run"].astype(str).tolist():
            baseline = float(obj50[obj50["run"].astype(str) == "pretrained"][metric].iloc[0])
            ax.axhline(baseline, color="black", linestyle="--", linewidth=1, label="pretrained")
        ax.set_title(title)
        ax.set_xlabel("Draft")
        ax.grid(axis="y", alpha=0.25)
        if metric == "speedup":
            ax.axhline(1.0, color="red", linestyle=":", linewidth=1, label="1.0x")
        ax.legend(loc="best")
    plt.tight_layout()
else:
    print("No completed 50k objective summaries found.")

In [ ]:
if not obj50.empty:
    best_acceptance = obj50.loc[obj50["acceptance_rate"].astype(float).idxmax()]
    best_speedup = obj50.loc[obj50["speedup"].astype(float).idxmax()]
    print(
        f"Best acceptance among completed 50k runs: {best_acceptance['run']} "
        f"({best_acceptance['acceptance_rate']:.3f})."
    )
    print(
        f"Best measured HF-loop speedup among completed 50k runs: {best_speedup['run']} "
        f"({best_speedup['speedup']:.3f}x)."
    )
    print(
        "Interpretation: KD can improve draft-target agreement, but our current "
        "instrumented HF speculative-decoding loop still has sub-1.0 wall-clock speedup "
        "because draft overhead and Python/cache-management overhead are not fully amortized."
    )

## 6. Qualitative Comparison from Generated Outputs

This cell reads the first available row from each `generations.jsonl` file. It is not a quality benchmark, but it confirms that the saved eval artifacts contain concrete generations for the same evaluation protocol.

In [ ]:
GEN_PATHS = {
    name: path.parent / "generations.jsonl"
    for name, path in SUMMARY_PATHS.items()
    if name in ["pretrained", "fkl", "rkl", "jsd"]
}

for name, path in GEN_PATHS.items():
    print("=" * 80)
    print(name, path)
    if not path.exists():
        print("missing generations.jsonl")
        continue
    with path.open("r", encoding="utf-8") as f:
        row = json.loads(f.readline())
    print("prompt:", row.get("prompt", "")[:400].replace("\n", " "))
    print("generation:", row.get("generation", "")[:700].replace("\n", " "))
    print("accepted_lens sample:", row.get("accepted_lens", [])[:20])

## 7. Reproducibility Commands

The commands below reproduce the 50k/8000-step evaluation runs from the interactive RunAI container. They are printed by default; set `RUN_EVAL=True` in Section 1 to execute them.

In [ ]:
def run_cmd(cmd: str, *, execute: bool = False) -> None:
    print("\n$ " + cmd)
    if execute:
        subprocess.run(cmd, shell=True, check=True, cwd=str(REPO_ROOT))

def eval_command(name: str, draft: str) -> str:
    return " ".join([
        "python scripts/evaluate_sd.py",
        f"data={DATASET}",
        "seed=42",
        f"run_name=eval_{name}_{RUN_SUFFIX}",
        f"results_dir={RESULTS_DIR / ('eval_' + name + '_' + RUN_SUFFIX)}",
        f"draft={draft}",
        f"prompts.jsonl={EVAL_JSONL}",
        f"prompts.limit={EVAL_LIMIT}",
        f"hf_cache={HF_CACHE}",
        f"model.target={TARGET_ID}",
        f"model.draft_default={PRETRAINED_DRAFT_ID}",
        "model.device=cuda",
        "model.dtype=bfloat16",
        "model.attn_impl=sdpa",
        "model.trust_remote_code=false",
        f"runtime.mode={RUNTIME_MODE}",
        f"runtime.temperature={RUNTIME_TEMPERATURE}",
        f"runtime.top_p={TOP_P}",
        f"runtime.gamma={GAMMA}",
        f"runtime.max_new_tokens={MAX_NEW_TOKENS}",
        f"eval.n_warmup={N_WARMUP}",
        f"eval.n_repeats={N_REPEATS}",
        "eval.run_vanilla_baseline=true",
        "eval.write_generations=true",
        "wandb.enabled=false",
    ])

drafts = {
    "pretrained": PRETRAINED_DRAFT_ID,
    "fkl": str(CHECKPOINTS_DIR / "fkl_ultra50k_s8000_seq512_a1_temp2" / "model"),
    "rkl": str(CHECKPOINTS_DIR / "rkl_ultra50k_s8000_seq512_a1_temp2" / "model"),
    "jsd": str(CHECKPOINTS_DIR / "jsd_ultra50k_s8000_seq512_a1_temp2" / "model"),
}

for name, draft in drafts.items():
    run_cmd(eval_command(name, draft), execute=RUN_EVAL)

## 8. Preliminary Takeaways

The current milestone results support the central hypothesis from our proposal: acceptance rate is useful but not sufficient for predicting wall-clock speedup. KD-trained drafts can improve target-draft agreement, but our measured HF-loop speedups remain below 1.0x because the current implementation is an instrumented research loop rather than an optimized production inference backend. This motivates the final-stage runtime sweep over `gamma`, decoding mode, and generation length, together with a controlled comparison of KD objectives and data scale.